# Evaluation Results Viewer
Reads one or more `eval_results.json` files (produced by `tools/test_tracked.py`) and presents the data as pandas DataFrames.

In [ ]:
import hashlib
import json
import os

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

# ── Configuration ────────────────────────────────────────────────────────────
# One path or a list of paths. Each file has the same nested JSON structure and
# holds either raw runs or runs from exactly one post-processing strategy. Name
# post-processed files `..._post_processed_<strategy>.json` (e.g.
# `..._post_processed_smoothnet_ws8.json`) so each strategy gets its own marker
# shape when plotted alongside the raw results and any other strategies.
RESULTS_FILES = [
    "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_e2e.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260610_coco_topdown.json",
    "/local/home/nkoefarago/mmpose/benchmark/results/20260815_coco_e2e.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260610_crowdpose_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260815_crowdpose_e2e.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260611_ochuman_topdown.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_3dpw_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_3dpw_topdown.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_emdb_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260715_emdb_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260815_emdb_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260806_emdb_video.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260816_emdb_tracking.json",

    # "/local/home/nkoefarago/mmpose/benchmark/results/20260727_posetrack21_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260727_posetrack21_topdown.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260815_posetrack21_e2e.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260806_posetrack21_video.json",
    # "/local/home/nkoefarago/mmpose/benchmark/results/20260816_posetrack21_tracking.json",
]

if isinstance(RESULTS_FILES, str):
    RESULTS_FILES = [RESULTS_FILES]

# ── Post-processing strategy per file ───────────────────────────────────────
# Each results file holds either raw runs or runs from exactly one post-processing
# strategy. The strategy name is used to pick a distinct marker shape per file, so
# any number of post-processing strategies (each in its own file) can be overlaid
# on the same plot alongside the raw (non post-processed) results.
RAW_STRATEGY = 'raw'
DEFAULT_MARKER = 'o'
# Cycled (in order of first appearance) across post-processing strategies found in
# the loaded data; extend if you load more than 9 distinct strategies at once.
STRATEGY_MARKER_POOL = ['*', 'X', '^', 'P', 'h', 'v', 'D', 's', '8']


def infer_strategy_label(path: str, is_post_processed: bool) -> str:
    """Post-processing strategy name for a results file (used for marker + legend).

    Every results file is assumed to hold either raw runs or runs from exactly one
    post-processing strategy. Prefers the convention
    `..._post_processed_<strategy>.json` -> 'strategy'; otherwise falls back to the
    filename stem so every distinct post-processed file still gets its own marker.
    """
    if not is_post_processed:
        return RAW_STRATEGY
    stem = os.path.splitext(os.path.basename(path))[0]
    tag = '_post_processed_'
    if tag in stem:
        return stem.split(tag, 1)[1]
    if stem.startswith('results_'):
        stem = stem[len('results_'):]
    return stem


def load_eval_results(path):
    with open(path) as f:
        raw = json.load(f)
    records = []
    for model_name, variants in raw.items():
        for variant, runs in variants.items():
            for run in runs:
                post_processed = bool(run.get('post_processed', False))
                record = {
                    'model': model_name,
                    'variant': variant,
                    'timestamp': pd.Timestamp(run['timestamp']),
                    'config': run.get('config', ''),
                    'checkpoint': run.get('checkpoint', ''),
                    'post_processed': post_processed,
                    'strategy': infer_strategy_label(path, post_processed),
                    'source_file': path,
                }
                record.update(run.get('metrics', {}))
                records.append(record)
    return records


# ── Load & flatten ───────────────────────────────────────────────────────────
records = []
for path in RESULTS_FILES:
    if not os.path.isfile(path):
        raise FileNotFoundError(f'Results file not found: {path}')
    records.extend(load_eval_results(path))

df = pd.DataFrame(records)

# Derive all metric columns (everything after the fixed columns)
ID_COLS = ["model", "variant", "timestamp"]
META_COLS = ['config', 'checkpoint', 'post_processed', 'strategy', 'source_file']
METRIC_COLS = [c for c in df.columns if c not in (ID_COLS + META_COLS)]


def strategy_sort_key(strategy: str):
    """Sort key placing 'raw' first, then strategies alphabetically."""
    return (strategy != RAW_STRATEGY, strategy)


def _build_strategy_markers(strategies):
    markers = {RAW_STRATEGY: DEFAULT_MARKER}
    others = sorted({s for s in strategies if s != RAW_STRATEGY})
    for i, strategy in enumerate(others):
        markers[strategy] = STRATEGY_MARKER_POOL[i % len(STRATEGY_MARKER_POOL)]
    return markers


# Fixed marker per strategy across all plots (order of first appearance in the
# full loaded dataset), so the same strategy always renders with the same shape.
STRATEGY_MARKERS = _build_strategy_markers(df['strategy'].unique()) if not df.empty else {}


def marker_for_strategy(strategy: str) -> str:
    return STRATEGY_MARKERS.get(strategy, DEFAULT_MARKER)

# Fixed plot color per known model family. tab20 alone only has 20 colors, so
# tab20b/tab20c are concatenated (60 total); extra HSV hues if that is still short.
# First 20 families keep their original tab20 colors. Extend order when adding families.
_MODEL_COLOR_ORDER = [
    'DARK', 'HRFormer', 'HRNet', 'MSPN', 'PCT', 'PETR', 'RF-DETR-Pose', 'RSN',
    'RTMPose', 'Sapiens', 'SimCC', 'UDP', 'ViTPose', 'YOLO-Pose', 'YOLO26-Pose',
    'Poseidon', 'TAR-ViTPose', 'PAVE-Net', 'DETRPose', 'GroupPose', 'QueryPose',
    'QueryPose-light', 'ED-Pose', 'AlphaPose', 'OpenPifPaf',
]


def _qualitative_palette(n):
    colors = []
    for name in ('tab20', 'tab20b', 'tab20c'):
        colors.extend(plt.get_cmap(name).colors)
    if len(colors) < n:
        extra = n - len(colors)
        colors.extend(plt.get_cmap('hsv')((i + 0.5) / extra) for i in range(extra))
    return list(colors[:n])


# Extra slots so unknown families hash into leftover qualitative colors, not
# back onto a reserved family.
_FALLBACK_SLOTS = 20
_model_palette = _qualitative_palette(len(_MODEL_COLOR_ORDER) + _FALLBACK_SLOTS)
MODEL_COLORS = {m: _model_palette[i] for i, m in enumerate(_MODEL_COLOR_ORDER)}

_TEMPORAL_PREFIXES = ('emdb', 'temporal', '3dpw', 'posetrack21')


def metric_label(col: str) -> str:
    """Short label for a metric column (strip emdb/ or temporal/ prefix)."""
    for prefix in _TEMPORAL_PREFIXES:
        prefix_slash = f'{prefix}/'
        if col.startswith(prefix_slash):
            return col[len(prefix_slash):]
    return col


def resolve_temporal_metric_col(df, suffix: str, prefixes=_TEMPORAL_PREFIXES):
    """Return emdb/* or temporal/* column for a metric suffix, preferring populated columns."""
    candidates = [
        f'{prefix}/{suffix}' for prefix in prefixes
        if f'{prefix}/{suffix}' in df.columns
    ]
    if not candidates:
        return None
    populated = [c for c in candidates if df[c].notna().any()]
    return populated[0] if populated else candidates[0]


def resolve_temporal_metric_cols(df, suffixes, prefixes=_TEMPORAL_PREFIXES):
    """Resolve multiple temporal metric columns (emdb/* or temporal/*)."""
    cols = []
    for suffix in suffixes:
        col = resolve_temporal_metric_col(df, suffix, prefixes)
        if col is not None:
            cols.append(col)
    return cols


def color_for_model(model: str):
    """Fixed color for known families; stable hash fallback for others."""
    if model in MODEL_COLORS:
        return MODEL_COLORS[model]
    digest = hashlib.md5(model.encode()).hexdigest()
    n_reserved = len(_MODEL_COLOR_ORDER)
    n_fallback = len(_model_palette) - n_reserved
    idx = n_reserved + (int(digest, 16) % n_fallback)
    return _model_palette[idx]


def scatter_runs(ax, sub, x_col, y_col, model, *, show_model_label=False):
    """Scatter points for one model; marker shape encodes post-processing strategy."""
    color = color_for_model(model)
    strategies = sorted(sub['strategy'].dropna().unique(), key=strategy_sort_key)
    for i, strategy in enumerate(strategies):
        strategy_sub = sub[sub['strategy'] == strategy]
        if strategy_sub.empty:
            continue
        ax.scatter(
            strategy_sub[x_col],
            strategy_sub[y_col],
            s=60 if strategy == RAW_STRATEGY else 140,
            alpha=0.85,
            color=color,
            marker=marker_for_strategy(strategy),
            label=model if (show_model_label and i == 0) else None,
        )


def strategy_shape_legend_handles(strategies):
    """Legend entries mapping marker shape to post-processing strategy."""
    handles = []
    for strategy in sorted(set(strategies), key=strategy_sort_key):
        handles.append(Line2D(
            [0], [0],
            marker=marker_for_strategy(strategy),
            color='0.45',
            linestyle='None',
            markersize=8 if strategy == RAW_STRATEGY else 12,
            label='raw' if strategy == RAW_STRATEGY else strategy,
        ))
    return handles


FPS_COL = 'perf/e2e/fps'
E2E_LATENCY_COL = 'perf/e2e/latency_ms_per_frame'
POSTPROC_LATENCY_COL = 'perf/postproc/latency_ms_per_frame'
COMBINED_FPS_COL = 'perf/combined/fps'


def add_throughput_fps(data, include_postproc_time=False):
    """Copy *data* and add ``COMBINED_FPS_COL`` as plot throughput (FPS).

    Default (``include_postproc_time=False``): the base model's e2e FPS
    (``perf/e2e/fps``), including for post-processed runs — post-processing
    does not change the pose model's inference speed.

    With ``include_postproc_time=True``: combined model + post-processing
    FPS, ``1000 / (e2e_ms + postproc_ms)``. Raw runs with no postproc
    latency keep the base-model FPS.
    """
    out = data.copy()
    if not include_postproc_time:
        out[COMBINED_FPS_COL] = (
            out[FPS_COL] if FPS_COL in out.columns
            else pd.Series(index=out.index, dtype=float)
        )
        return out

    if E2E_LATENCY_COL in out.columns:
        e2e_ms = out[E2E_LATENCY_COL].astype(float)
    else:
        e2e_ms = pd.Series(index=out.index, dtype=float)
    if FPS_COL in out.columns:
        from_fps = 1000.0 / out[FPS_COL].where(out[FPS_COL] > 0)
        e2e_ms = e2e_ms.fillna(from_fps)

    if POSTPROC_LATENCY_COL in out.columns:
        pp_ms = out[POSTPROC_LATENCY_COL].astype(float).fillna(0.0)
    else:
        pp_ms = 0.0
    total_ms = e2e_ms + pp_ms
    out[COMBINED_FPS_COL] = 1000.0 / total_ms.where(total_ms > 0)
    return out


def latest_plot_runs(data):
    """Latest run per (model, variant, strategy) for scatter plots."""
    return (
        data.sort_values('timestamp')
        .groupby(['model', 'variant', 'strategy'], sort=False)
        .last()
        .reset_index()
    )


def filter_by_model_variants(data, pairs=None):
    """Keep rows whose (model, variant) is in *pairs*. None keeps everything."""
    if pairs is None:
        return data.reset_index(drop=True)
    allowed = set(pairs)
    mask = data.apply(lambda r: (r['model'], r['variant']) in allowed, axis=1)
    return data[mask].reset_index(drop=True)


def available_model_variant_pairs(data):
    """Distinct (model, variant) pairs in *data* (latest run per post_processed flag)."""
    return (
        latest_plot_runs(data)[['model', 'variant']]
        .drop_duplicates()
        .sort_values(['model', 'variant'])
        .reset_index(drop=True)
    )


def _axis_limit(lo, hi, *, padding, clamp_hi=None):
    span = hi - lo
    if span <= 0:
        span = abs(hi) or abs(lo) or 1.0
    lo_p = lo - span * padding
    hi_p = hi + span * padding
    if clamp_hi is not None:
        hi_p = min(clamp_hi, hi_p)
    return (max(0, lo_p), hi_p)


def compute_axis_limits_from_data(data, *, fps_col=FPS_COL, padding=0.05):
    """Compute shared axis limits from latest runs in *data*."""
    latest = latest_plot_runs(data)
    limits = {}
    if fps_col in latest.columns and latest[fps_col].notna().any():
        vals = latest[fps_col].dropna()
        limits['fps'] = _axis_limit(vals.min(), vals.max(), padding=padding)
    for col in ('coco/AP', 'coco/AR'):
        if col in latest.columns and latest[col].notna().any():
            vals = latest[col].dropna()
            limits[col] = _axis_limit(vals.min(), vals.max(), padding=padding, clamp_hi=1.0)
    for suffix in ('bMPJAE', 'bMPJVE', 'tMPJAE', 'tMPJVE'):
        col = resolve_temporal_metric_col(latest, suffix)
        if col is not None and latest[col].notna().any():
            vals = latest[col].dropna()
            limits[suffix] = _axis_limit(vals.min(), vals.max(), padding=padding)
    hota_col = 'tracking/HOTA'
    if hota_col in latest.columns and latest[hota_col].notna().any():
        vals = latest[hota_col].dropna()
        limits['HOTA'] = _axis_limit(
            vals.min(), vals.max(), padding=padding, clamp_hi=1.0)
    return limits


def build_plot_axis_limits(reference_data, manual_limits, *, padding=0.05):
    """Merge auto-computed limits from *reference_data* with manual overrides."""
    computed = compute_axis_limits_from_data(reference_data, padding=padding)
    keys = (
        'fps', 'coco/AP', 'coco/AR', 'bMPJAE', 'bMPJVE', 'tMPJAE', 'tMPJVE',
        'HOTA',
    )
    return {
        key: manual_limits.get(key)
        if manual_limits.get(key) is not None
        else computed.get(key)
        for key in keys
    }


def apply_plot_axis_limits(ax, x_col, y_col, limits, *, fps_col=FPS_COL):
    """Apply shared axis limits configured for scatter plots."""
    if limits.get('fps') and x_col == fps_col:
        ax.set_xlim(limits['fps'])
    for y_key in (y_col, metric_label(y_col), y_col.rsplit('/', 1)[-1]):
        if limits.get(y_key):
            ax.set_ylim(limits[y_key])
            break


print(
    f'Loaded {len(df)} evaluation run(s) from {len(RESULTS_FILES)} file(s) '
    f'| metrics: {METRIC_COLS}'
)

## All evaluation entries

In [ ]:
display_cols = ID_COLS + METRIC_COLS

all_entries = (
    df[display_cols]
    .sort_values(['model', 'variant', 'timestamp'])
    .reset_index(drop=True)
)

display(
    all_entries.style
    .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
    .set_caption('All evaluation runs')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Latest result per model / variant

In [ ]:
DISPLAY_COLS = ["model", "variant"] + METRIC_COLS

AP_AR_COLS = [c for c in METRIC_COLS if '/AP' in c or '/AR' in c]
OTHER_METRIC_COLS = [c for c in METRIC_COLS if c not in AP_AR_COLS]

fmt = {c: (lambda x: f'{x*100:.1f}' if pd.notna(x) else '—') for c in AP_AR_COLS}
fmt.update({c: '{:.4f}' for c in OTHER_METRIC_COLS})

latest = (
    df.sort_values('timestamp')
    .groupby(['model', 'variant'], sort=False)
    .last()
    .reset_index()
)[DISPLAY_COLS].sort_values(['model', 'variant']).reset_index(drop=True)

display(
    latest.style
    .format(fmt, na_rep='—')
    .set_caption('Latest result per model / variant')
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
)

## Best `coco/AP` run per model

In [ ]:
AP_COL = 'coco/AP'

if AP_COL not in df.columns:
    print(f"Column '{AP_COL}' not found in results. "
          "Available metric columns:", METRIC_COLS)
else:
    best_ap = (
        df.dropna(subset=[AP_COL])
        .sort_values(AP_COL, ascending=False)
        .groupby('model', sort=False)
        .first()
        .reset_index()
    )[ID_COLS + METRIC_COLS].sort_values('model').reset_index(drop=True)

    display(
        best_ap.style
        .format({c: '{:.4f}' for c in METRIC_COLS}, na_rep='—')
        .set_caption(f'Best {AP_COL} run per model (variant + timestamp shown)')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'), ('font-weight', 'bold')]}])
    )

## Plot filter & fixed axis limits

Define named groups as hand-picked `(model, variant)` lists, set `ACTIVE_GROUP`, and lock axis ranges so different groups share the same plot scale.

Re-run this cell after changing `MODEL_GROUPS`, `ACTIVE_GROUP`, or axis settings. Downstream plot cells use `df_plot` and `PLOT_AXIS_LIMITS`.

In [ ]:
# ── Model groups (hand-defined) ───────────────────────────────────────────────
# Each group is a list of (model, variant) tuples. Set SHOW_AVAILABLE_PAIRS = True
# once to list every pair in the loaded data, then copy entries into MODEL_GROUPS.
SHOW_AVAILABLE_PAIRS = False

MODEL_GROUPS = {
    'light': [
        ('YOLO-Pose', 'large'),
        ('YOLO-Pose', 'medium'),
        ('YOLO-Pose', 'small'),
        ('YOLO-Pose', 'tiny'),
        ('YOLO26-Pose', 'xlarge'),
        ('YOLO26-Pose', 'large'),
        ('YOLO26-Pose', 'medium'),
        ('YOLO26-Pose', 'small'),
        ('YOLO26-Pose', 'nano'),
        ('RF-DETR-Pose', 'xlarge'),
        ('ViTPose', 'small-rfdetr'),
        ('UDP', 'hrnetw32-li-rtmdet'),
        ('UDP', 'cspnext-tiny-rfdetr'),
        ('RTMPose', 'medium-rfdetr'),
        ('RTMPose', 'small-rfdetr'),
        ('RTMPose', 'tiny-rfdetr'),
        ('SimCC', 'resnet50-si-rfdetr'),
        ('DETRPose', 'n'),
        ('DETRPose', 's'),
        ('DETRPose', 'm'),
        ('DETRPose', 'l'),
        ('DETRPose', 'x'),
    ],
    'heavy': [
        ('ViTPose', 'large-rfdetr'),
        ('ViTPose', 'huge-rfdetr'),
        ('UDP', 'hrnetw48-li-rtmdet'),
        ('SimCC', 'resnet50-li-rtmdet'),
        ('RTMPose', 'large-rtmdet'),
        ('PETR', 'swin-l'),
        ('PETR', 'resnet101'),
        ('PETR', 'resnet50'),
        ('PCT', 'base-rtmdet'),
        ('PCT', 'large-rtmdet'),
        ('PCT', 'huge-rtmdet'),
        ('MSPN', '4xresnet50-rtmdet'),
        ('HRNet', 'hrnetw48-si-rtmdet'),
        ('HRFormer', 'base-si-rtmdet'),
        ('DARK', 'hrnetw48-si-rtmdet'),
        ("Poseidon", "vits-posetrack21-rfdetr"),
        ("Poseidon", "vitb-posetrack21-rfdetr"),
        ("Poseidon", "vith-posetrack21-rfdetr"),
        ("TAR-ViTPose", "vitb-posetrack17-rfdetr"),
        ("TAR-ViTPose", "vith-posetrack17-rfdetr"),
        ("PAVE-Net", "r50-posetrack17"),
    ],
}
ACTIVE_GROUP = None

# ── Fixed axis limits ────────────────────────────────────────────────────────
# Limits are computed from the reference dataset, then manual overrides win.
# Use AXIS_LIMITS_SOURCE='full' so filtered plots keep the same axes as the
# unfiltered baseline (recommended for cross-group comparison).
AXIS_LIMITS_SOURCE = 'full'  # 'full' | 'filtered'
AXIS_PADDING = 0.05
MANUAL_AXIS_LIMITS = {
    'fps': None,
    'coco/AP': None,
    'coco/AR': None,
    'bMPJAE': None,
    'bMPJVE': None,
    'tMPJAE': None,
    'tMPJVE': None,
    'HOTA': None,
}

if SHOW_AVAILABLE_PAIRS:
    display(
        available_model_variant_pairs(df)
        .style.set_caption('Available (model, variant) pairs — copy into MODEL_GROUPS')
    )

_selected_pairs = MODEL_GROUPS.get(ACTIVE_GROUP) if ACTIVE_GROUP else None
if ACTIVE_GROUP and _selected_pairs is None:
    raise KeyError(
        f'ACTIVE_GROUP={ACTIVE_GROUP!r} not in MODEL_GROUPS. '
        f'Available: {sorted(MODEL_GROUPS)}'
    )

df_plot = filter_by_model_variants(df, _selected_pairs)

_ref_for_limits = df if AXIS_LIMITS_SOURCE == 'full' else df_plot
PLOT_AXIS_LIMITS = build_plot_axis_limits(
    _ref_for_limits, MANUAL_AXIS_LIMITS, padding=AXIS_PADDING,
)

_group_label = ACTIVE_GROUP or 'all'
_latest_plot = latest_plot_runs(df_plot)
print(
    f'Plot filter: {_group_label} | '
    f'{len(df_plot)} run(s), {len(_latest_plot)} latest point(s) '
    f'| {df_plot["model"].nunique()} model(s), '
    f'{df_plot.groupby(["model", "variant"]).ngroups} model-variant pair(s)'
)
if ACTIVE_GROUP and _selected_pairs:
    _missing = set(_selected_pairs) - set(
        zip(df_plot['model'], df_plot['variant'])
    )
    if _missing:
        print('Warning: group entries not found in loaded data:', sorted(_missing))

# if df_plot.empty:
#     print('Warning: filter matched no runs — plots below will be empty.')
# else:
#     display(
#         _latest_plot[['model', 'variant', 'post_processed']]
#         .drop_duplicates()
#         .sort_values(['model', 'variant', 'post_processed'])
#         .reset_index(drop=True)
#         .style.set_caption(f'Included entries ({_group_label})')
#     )

_active_limits = {k: v for k, v in PLOT_AXIS_LIMITS.items() if v is not None}
print('Fixed axis limits:', _active_limits or '(auto per plot — set MANUAL_AXIS_LIMITS or load data)')

## AP vs e2e FPS

In [ ]:
AP_COL = 'coco/AP'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AP points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AP_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[AP_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {AP_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, AP_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AP_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[AP_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('coco/AP')
        ax.set_title('Precision vs end-to-end throughput (latest run per model / variant)')
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, AP_COL, PLOT_AXIS_LIMITS)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + strategy_shape_legend_handles(plot_df['strategy'].unique()),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, AP_COL]]
        #     .sort_values([AP_COL, FPS_COL], ascending=[False, False])
        #     .reset_index(drop=True)
        #     .style.format({AP_COL: '{:.3f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
       
        # )
        plt.show()

## AR vs e2e FPS

In [ ]:
AR_COL = 'coco/AR'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N highest-AR points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (AR_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[AR_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {AR_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, AR_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, AR_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[AR_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('coco/AR')
        ax.set_title('Recall vs end-to-end throughput (latest run per model / variant)')
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, AR_COL, PLOT_AXIS_LIMITS)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + strategy_shape_legend_handles(plot_df['strategy'].unique()),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # Variant names for each point (no overlap on the chart).
        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, AR_COL]]
        #     .sort_values([AR_COL, FPS_COL], ascending=[False, False])
        #     .reset_index(drop=True)
        #     .style.format({AR_COL: '{:.3f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
       
        # )
        plt.show()

## EMDB temporal metric vs e2e FPS

In [ ]:
METRIC_COL_NAME = resolve_temporal_metric_col(df_plot, 'bMPJVE')
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N best (lowest) points (0 = no point labels).
ANNOTATE_TOP_N = 0

if METRIC_COL_NAME is None:
    print('No bMPJVE column found (expected emdb/bMPJVE or temporal/bMPJVE).')
    print('Available metric columns:', METRIC_COLS)
elif FPS_COL not in df_plot.columns:
    print(f'Missing column for plot: {FPS_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[METRIC_COL_NAME, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {METRIC_COL_NAME} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, METRIC_COL_NAME, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nsmallest(ANNOTATE_TOP_N, METRIC_COL_NAME)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[METRIC_COL_NAME]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel(metric_label(METRIC_COL_NAME))
        ax.set_title(
            f'{metric_label(METRIC_COL_NAME)} vs end-to-end throughput '
            '(latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, METRIC_COL_NAME, PLOT_AXIS_LIMITS)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + strategy_shape_legend_handles(plot_df['strategy'].unique()),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, METRIC_COL_NAME]]
        #     .sort_values([METRIC_COL_NAME, FPS_COL], ascending=[True, False])
        #     .reset_index(drop=True)
        #     .style.format({METRIC_COL_NAME: '{:.4f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
        # )
        plt.show()

## HOTA vs inference speed

Default x-axis is the **base model's** e2e FPS (`perf/e2e/fps`), including for post-processed runs. Set `INCLUDE_POSTPROC_TIME = True` to plot combined model + post-processing throughput instead (`1000 / (e2e_ms + postproc_ms)`).

In [ ]:
HOTA_COL = 'tracking/HOTA'
# False: x-axis is the base model's e2e FPS (perf/e2e/fps), including for
# post-processed runs. True: x-axis is combined model + post-processing FPS
# (1000 / (e2e_ms + postproc_ms)); raw runs without postproc latency are unchanged.
INCLUDE_POSTPROC_TIME = False
# Set to a positive int to label only the N highest-HOTA points (0 = no point labels).
ANNOTATE_TOP_N = 0

if HOTA_COL not in df_plot.columns:
    print(f'Missing column for plot: {HOTA_COL}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = add_throughput_fps(
        latest_plot_runs(df_plot),
        include_postproc_time=INCLUDE_POSTPROC_TIME,
    )
    x_col = COMBINED_FPS_COL
    plot_df = plot_df.dropna(subset=[HOTA_COL, x_col])

    if plot_df.empty:
        print(f'No rows with both {HOTA_COL} and throughput FPS.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, x_col, HOTA_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nlargest(ANNOTATE_TOP_N, HOTA_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[x_col], row[HOTA_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        if INCLUDE_POSTPROC_TIME:
            xlabel = 'combined FPS (model + post-processing)'
            title_fps = 'combined model + post-processing throughput'
        else:
            xlabel = 'base-model e2e FPS'
            title_fps = 'base-model throughput'

        ax.set_xlabel(xlabel)
        ax.set_ylabel('HOTA')
        ax.set_title(
            f'HOTA vs {title_fps} (latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)

        limits = dict(PLOT_AXIS_LIMITS)
        if INCLUDE_POSTPROC_TIME and plot_df[x_col].notna().any():
            vals = plot_df[x_col].dropna()
            limits['fps'] = _axis_limit(
                vals.min(), vals.max(), padding=AXIS_PADDING)
        apply_plot_axis_limits(ax, x_col, HOTA_COL, limits, fps_col=x_col)

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + strategy_shape_legend_handles(plot_df['strategy'].unique()),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()
        plt.show()


## ID switches per 100 frames vs e2e FPS

In [ ]:
IDSW_COL = 'tracking/IDSwitchesPer100Frames'
FPS_COL = 'perf/e2e/fps'
# Set to a positive int to label only the N best (fewest ID switches) points (0 = no point labels).
ANNOTATE_TOP_N = 0

missing = [c for c in (IDSW_COL, FPS_COL) if c not in df_plot.columns]
if missing:
    print(f'Missing columns for plot: {missing}')
    print('Available metric columns:', METRIC_COLS)
else:
    plot_df = latest_plot_runs(df_plot).dropna(subset=[IDSW_COL, FPS_COL])

    if plot_df.empty:
        print(f'No rows with both {IDSW_COL} and {FPS_COL}.')
    else:
        models = plot_df['model'].unique()

        fig, ax = plt.subplots(figsize=(15, 9))
        for model in models:
            sub = plot_df[plot_df['model'] == model]
            scatter_runs(ax, sub, FPS_COL, IDSW_COL, model, show_model_label=True)

        if ANNOTATE_TOP_N > 0:
            top = plot_df.nsmallest(ANNOTATE_TOP_N, IDSW_COL)
            for _, row in top.iterrows():
                ax.annotate(
                    f"{row['variant']}",
                    (row[FPS_COL], row[IDSW_COL]),
                    xytext=(4, 4),
                    textcoords='offset points',
                    fontsize=7,
                    color=color_for_model(row['model']),
                )

        ax.set_xlabel('e2e FPS')
        ax.set_ylabel('ID switches / 100 frames')
        ax.set_title(
            'ID switches per 100 frames vs end-to-end throughput '
            '(latest run per model / variant)'
        )
        ax.grid(True, alpha=0.3)
        apply_plot_axis_limits(ax, FPS_COL, IDSW_COL, PLOT_AXIS_LIMITS)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(
            handles=handles + strategy_shape_legend_handles(plot_df['strategy'].unique()),
            title='model',
            bbox_to_anchor=(1.02, 1),
            loc='upper left',
            fontsize=8,
            framealpha=0.9,
        )
        fig.tight_layout()

        # display(
        #     plot_df[['model', 'variant', 'post_processed', FPS_COL, IDSW_COL]]
        #     .sort_values([IDSW_COL, FPS_COL], ascending=[True, False])
        #     .reset_index(drop=True)
        #     .style.format({IDSW_COL: '{:.4f}', FPS_COL: '{:.1f}'})
        #     .set_caption('Points on plot (hover-free lookup)')
        # )
        plt.show()

## EMDB keypoint trajectory over time

Loads a benchmark prediction export (`manifest.json` + `frames.json` from `benchmark_e2e.py`) and plots one keypoint's **x** and **y** image coordinates across frames in a sequence.

In [ ]:
import json
import os.path as osp

# ── Configuration ────────────────────────────────────────────────────────────
# Directory with manifest.json and frames.json (benchmark_e2e export).
PRED_DIR = (
    '/local/home/nkoefarago/mmpose/'
    'benchmark/predictions/20260622_emdb_topdown/ViTPose-small-rfdetr__postproc'
)
SEQUENCE = None  # None = first sequence in the bundle, or e.g. 'P1/14_outdoor_climb'
KEYPOINT = 'right_wrist'  # name from dataset_meta, or int index (0–16 for COCO-17)
SHOW_GT = True  # overlay matched ground-truth keypoint
MIN_SCORE = 0.0  # skip pred points below this keypoint score


def _load_prediction_bundle(pred_dir: str):
    manifest_path = osp.join(pred_dir, 'manifest.json')
    frames_path = osp.join(pred_dir, 'frames.json')
    if not osp.isfile(manifest_path) or not osp.isfile(frames_path):
        raise FileNotFoundError(
            f'Expected manifest.json and frames.json in {pred_dir}')
    with open(manifest_path, encoding='utf-8') as f:
        manifest = json.load(f)
    with open(frames_path, encoding='utf-8') as f:
        frames = json.load(f)
    return manifest, frames


def _emdb_sequence_name(img_path: str) -> str:
    """P1/14_outdoor_climb/images/00042.jpg -> P1/14_outdoor_climb."""
    parts = img_path.replace('\\', '/').split('/')
    if len(parts) >= 3 and parts[-2] == 'images':
        return '/'.join(parts[:-2])
    return osp.dirname(img_path)


def _emdb_frame_index(img_path: str) -> int:
    basename = osp.splitext(osp.basename(img_path))[0]
    if basename.startswith('image_'):
        return int(basename.split('_')[-1])
    return int(basename)


def _keypoint_index(name_or_idx, dataset_meta: dict) -> int:
    if isinstance(name_or_idx, int):
        return name_or_idx
    id2name = dataset_meta.get('keypoint_id2name', {})
    for idx, name in id2name.items():
        if name == name_or_idx:
            return int(idx)
    for idx, info in dataset_meta.get('keypoint_info', {}).items():
        if info.get('name') == name_or_idx:
            return int(idx)
    names = list(id2name.values()) or [
        info.get('name') for info in dataset_meta.get('keypoint_info', {}).values()
    ]
    raise ValueError(
        f'Keypoint {name_or_idx!r} not found. Available: {names}')


def _pick_instance_idx(instances, matches, role: str):
    idx_key = 'pred_idx' if role == 'pred' else 'gt_idx'
    if matches:
        return matches[0][idx_key]
    if instances:
        return 0
    return None


def _extract_keypoint_trajectory(
    frames,
    sequence: str,
    kpt_idx: int,
    show_gt: bool,
    min_score: float,
):
    rows = []
    for frame in frames:
        if _emdb_sequence_name(frame['img_path']) != sequence:
            continue

        matches = frame['metrics'].get('matches', [])
        pred_instances = frame['predictions']['instances']
        gt_instances = frame['ground_truth']['instances']

        pred_i = _pick_instance_idx(pred_instances, matches, 'pred')
        if pred_i is None:
            continue

        pred_kpt = pred_instances[pred_i]['keypoints'][kpt_idx]
        pred_score = pred_instances[pred_i].get('keypoint_scores', [1.0])[kpt_idx]
        if pred_score < min_score:
            continue

        ori_shape = frame.get('ori_shape', [0, 0])
        img_h, img_w = int(ori_shape[0]), int(ori_shape[1])

        gt_kpt = None
        if show_gt:
            gt_i = _pick_instance_idx(gt_instances, matches, 'gt')
            if gt_i is not None:
                gt_kpt = gt_instances[gt_i]['keypoints'][kpt_idx]

        rows.append({
            'frame': _emdb_frame_index(frame['img_path']),
            'pred_x': pred_kpt[0],
            'pred_y': pred_kpt[1],
            'pred_score': pred_score,
            'gt_x': gt_kpt[0] if gt_kpt is not None else None,
            'gt_y': gt_kpt[1] if gt_kpt is not None else None,
            'img_h': img_h,
            'img_w': img_w,
        })

    return pd.DataFrame(rows).sort_values('frame').reset_index(drop=True)


pred_dir = osp.abspath(PRED_DIR)
manifest, frames = _load_prediction_bundle(pred_dir)
dataset_meta = manifest.get('dataset_meta', {})
kpt_idx = _keypoint_index(KEYPOINT, dataset_meta)
kpt_name = (
    KEYPOINT if isinstance(KEYPOINT, str)
    else dataset_meta.get('keypoint_id2name', {}).get(str(KEYPOINT), str(KEYPOINT))
)

sequences = sorted({_emdb_sequence_name(f['img_path']) for f in frames})
if not sequences:
    raise ValueError(f'No frames found in {pred_dir}')

sequence = SEQUENCE or sequences[0]
if sequence not in sequences:
    raise ValueError(
        f'Sequence {sequence!r} not in bundle. Available ({len(sequences)}): '
        f'{sequences[:10]}{"..." if len(sequences) > 10 else ""}'
    )

traj = _extract_keypoint_trajectory(
    frames, sequence, kpt_idx, SHOW_GT, MIN_SCORE)

if traj.empty:
    print(f'No trajectory points for sequence {sequence!r} '
          f'(keypoint={kpt_name}, min_score={MIN_SCORE}).')
else:
    model_label = manifest.get('model_name', 'model')
    if manifest.get('model_variant'):
        model_label = f'{model_label}-{manifest["model_variant"]}'

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    axes[0].plot(traj['frame'], traj['pred_x'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_x'].notna().any():
        axes[0].plot(
            traj['frame'], traj['gt_x'], color='C0', linestyle='--', label='ground truth')
    axes[0].set_ylabel('x (px)')
    axes[0].set_title(f'{kpt_name} — horizontal position')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc='upper right')

    axes[1].plot(traj['frame'], traj['pred_y'], color='C3', label='prediction')
    if SHOW_GT and traj['gt_y'].notna().any():
        axes[1].plot(
            traj['frame'], traj['gt_y'], color='C0', linestyle='--', label='ground truth')
    axes[1].set_ylabel('y (px)')
    axes[1].set_title(f'{kpt_name} — vertical position')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(loc='upper right')

    axes[2].plot(traj['frame'], traj['pred_score'], color='C2', label='confidence')
    axes[2].set_xlabel('frame index')
    axes[2].set_ylabel('score')
    axes[2].set_title(f'{kpt_name} — keypoint confidence')
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3)
    axes[2].legend(loc='upper right')

    fig.suptitle(
        f'{model_label} | {manifest.get("test_dataset", "emdb")} | {sequence} '
        f'({len(traj)} frames)',
        y=1.02,
    )
    fig.tight_layout()
    plt.show()

    display(
        traj.head(10).style.format({
            'pred_x': '{:.1f}',
            'pred_y': '{:.1f}',
            'pred_score': '{:.3f}',
            'gt_x': '{:.1f}',
            'gt_y': '{:.1f}',
        }).set_caption(f'First frames — {sequence}')
    )
    if len(sequences) > 1:
        print(f'Other sequences ({len(sequences)} total): {sequences}')

## Score vs. keypoint error (calibration)

Loads a benchmark prediction bundle (`manifest.json` + `frames.json`) and plots the **Euclidean distance between each predicted keypoint and its matched GT keypoint** against the **predicted keypoint score**.

A well-calibrated model should show monotonically *decreasing* error as the score increases (high score → high confidence → small error). The Spearman ρ per keypoint quantifies this: ρ < 0 is good.

In [ ]:
import json
import os.path as osp

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# ── Configuration ─────────────────────────────────────────────────────────────
# Directory produced by benchmark_e2e.py (must contain manifest.json + frames.json).
CALIB_PRED_DIR = (
    '/local/home/nkoefarago/mmpose/'
    'benchmark/predictions/20260715_emdb_e2e/YOLO26-Pose-large'
)
# Keypoints to include: list of names (str), list of indices (int), or None / 'all'.
CALIB_KEYPOINTS = None
# Normalize pixel distances by the geometric mean of image height × width.
NORMALIZE_BY_IMAGE_SIZE = False
# Number of score buckets for the reliability / calibration curve.
N_BINS = 20
# Bins with fewer samples than this threshold are hidden from the calibration curve.
MIN_SAMPLES_PER_BIN = 10
# Upper percentile used for the error-envelope shading on the calibration curve.
ERROR_PERCENTILE = 75
# Ignore keypoint predictions below this score (skip near-background detections).
MIN_SCORE = 0.0
# Distance values above this percentile are clipped in the density plot only.
MAX_DIST_PERCENTILE = 99


# ── Helpers ───────────────────────────────────────────────────────────────────
def _calib_load_bundle(pred_dir: str):
    for fname in ('manifest.json', 'frames.json'):
        p = osp.join(pred_dir, fname)
        if not osp.isfile(p):
            raise FileNotFoundError(f'Missing {fname} in {pred_dir}')
    with open(osp.join(pred_dir, 'manifest.json')) as f:
        manifest = json.load(f)
    with open(osp.join(pred_dir, 'frames.json')) as f:
        frames = json.load(f)
    return manifest, frames


def _calib_kpt_indices(kpts_cfg, dataset_meta: dict):
    """Return a list of keypoint indices, or None (= all keypoints)."""
    if kpts_cfg is None or kpts_cfg == 'all':
        return None
    id2name = dataset_meta.get('keypoint_id2name', {})
    name2id = {v: int(k) for k, v in id2name.items()}
    result = []
    for kpt in ([kpts_cfg] if not isinstance(kpts_cfg, (list, tuple)) else kpts_cfg):
        if isinstance(kpt, int):
            result.append(kpt)
        elif kpt in name2id:
            result.append(name2id[kpt])
        else:
            raise ValueError(f'Keypoint {kpt!r} not found. Available: {sorted(name2id)}')
    return result


def _calib_collect(frames, kpt_indices, normalize: bool, min_score: float):
    """Iterate over all matched (pred, GT) pairs and collect (score, distance, kpt_id)."""
    scores_out, dists_out, kpt_ids_out = [], [], []

    for frame in frames:
        matches = frame['metrics'].get('matches', [])
        pred_insts = frame['predictions']['instances']
        gt_insts   = frame['ground_truth']['instances']
        ori_shape  = frame.get('ori_shape', [1, 1])
        img_h, img_w = float(ori_shape[0]), float(ori_shape[1])
        scale = (img_h * img_w) ** 0.5 if normalize else 1.0

        for match in matches:
            pred_inst = pred_insts[match['pred_idx']]
            gt_inst   = gt_insts[match['gt_idx']]

            pred_kpts   = pred_inst['keypoints']
            pred_scores = pred_inst.get('keypoint_scores', [1.0] * len(pred_kpts))
            gt_kpts     = gt_inst['keypoints']
            gt_vis      = gt_inst.get('keypoints_visible', [1.0] * len(gt_kpts))

            n_kpts  = min(len(pred_kpts), len(gt_kpts))
            indices = kpt_indices if kpt_indices is not None else range(n_kpts)

            for k in indices:
                if k >= n_kpts:
                    continue
                vis = float(gt_vis[k]) if k < len(gt_vis) else 1.0
                if vis == 0:
                    continue
                score = float(pred_scores[k]) if k < len(pred_scores) else 1.0
                if score < min_score:
                    continue
                dx = float(pred_kpts[k][0]) - float(gt_kpts[k][0])
                dy = float(pred_kpts[k][1]) - float(gt_kpts[k][1])
                dist = (dx * dx + dy * dy) ** 0.5 / scale
                scores_out.append(score)
                dists_out.append(dist)
                kpt_ids_out.append(int(k))

    return (
        np.array(scores_out, dtype=np.float32),
        np.array(dists_out,  dtype=np.float32),
        np.array(kpt_ids_out, dtype=np.int32),
    )


# ── Load & collect ─────────────────────────────────────────────────────────────
_calib_dir  = osp.abspath(CALIB_PRED_DIR)
_manifest_c, _frames_c = _calib_load_bundle(_calib_dir)
_dmeta_c    = _manifest_c.get('dataset_meta', {})
_kpt_idx_c  = _calib_kpt_indices(CALIB_KEYPOINTS, _dmeta_c)

scores, dists, kpt_ids = _calib_collect(_frames_c, _kpt_idx_c, NORMALIZE_BY_IMAGE_SIZE, MIN_SCORE)

if len(scores) == 0:
    raise ValueError(
        f'No (score, distance) pairs collected from {_calib_dir}. '
        'Check matches / visibility / MIN_SCORE.'
    )

dist_clip   = np.percentile(dists, MAX_DIST_PERCENTILE)
dists_plot  = np.clip(dists, 0.0, dist_clip)
dist_unit   = 'normalized distance' if NORMALIZE_BY_IMAGE_SIZE else 'distance (px)'

# Score axis: use the observed range (some models emit unnormalized scores, e.g. PCT).
_score_lo = float(max(MIN_SCORE, scores.min()))
_score_hi = float(scores.max())
if _score_hi <= _score_lo:
    _score_hi = _score_lo + 1e-3
_score_pad = 0.02 * (_score_hi - _score_lo)
_score_xlim = (_score_lo - _score_pad, _score_hi + _score_pad)

_model_label_c = _manifest_c.get('model_name', 'model')
if _manifest_c.get('model_variant'):
    _model_label_c = f'{_model_label_c}-{_manifest_c["model_variant"]}'

print(f'Collected {len(scores):,} (score, distance) pairs from {len(_frames_c)} frames.')
print(f'Score range  : [{scores.min():.3f}, {scores.max():.3f}]')
print(f'Distance range: [{dists.min():.4f}, {dists.max():.4f}]  ({dist_unit})')
if scores.min() < 0 or scores.max() > 1.01:
    print(
        'Note: keypoint scores are outside [0, 1]; '
        'binning and x-limits use the observed score range.'
    )

# ── Binned calibration statistics ──────────────────────────────────────────────
_bin_edges   = np.linspace(_score_lo, _score_hi, N_BINS + 1)
_bin_centers = 0.5 * (_bin_edges[:-1] + _bin_edges[1:])
_bin_mean  = np.full(N_BINS, np.nan)
_bin_p25   = np.full(N_BINS, np.nan)
_bin_p_hi  = np.full(N_BINS, np.nan)
_bin_n     = np.zeros(N_BINS, dtype=int)

for _i, (_lo, _hi) in enumerate(zip(_bin_edges[:-1], _bin_edges[1:])):
    _mask_b = (scores >= _lo) & (scores <= _hi if _i == N_BINS - 1 else scores < _hi)
    _n = int(_mask_b.sum())
    _bin_n[_i] = _n
    if _n >= MIN_SAMPLES_PER_BIN:
        _d = dists[_mask_b]
        _bin_mean[_i] = _d.mean()
        _bin_p25[_i]  = np.percentile(_d, 25)
        _bin_p_hi[_i] = np.percentile(_d, ERROR_PERCENTILE)

_valid = ~np.isnan(_bin_mean)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, (ax_hex, ax_cal) = plt.subplots(1, 2, figsize=(16, 6))

# Left – hexbin density --------------------------------------------------------
hb = ax_hex.hexbin(scores, dists_plot, gridsize=60, cmap='YlOrRd',
                    mincnt=1, bins='log')
fig.colorbar(hb, ax=ax_hex, label='log₁₀(count)')

if _valid.any():
    ax_hex.plot(_bin_centers[_valid], _bin_mean[_valid],
                color='C0', linewidth=2, label='mean error per bin', zorder=5)
    ax_hex.fill_between(_bin_centers[_valid], _bin_p25[_valid], _bin_p_hi[_valid],
                         color='C0', alpha=0.25,
                         label=f'25th–{ERROR_PERCENTILE}th pct.')

ax_hex.set_xlabel('predicted score')
ax_hex.set_ylabel(f'{dist_unit}  (clipped at {MAX_DIST_PERCENTILE}th pct.)')
ax_hex.set_title(f'Score vs. keypoint error — {_model_label_c}\n({len(scores):,} keypoints)')
ax_hex.grid(True, alpha=0.3)
_hex_handles, _hex_labels = ax_hex.get_legend_handles_labels()
if _hex_handles:
    ax_hex.legend(fontsize=9)
ax_hex.set_xlim(*_score_xlim)
ax_hex.set_ylim(0, dist_clip * 1.05)

# Right – calibration curve + sample counts -----------------------------------
ax_twin = ax_cal.twinx()

ax_twin.bar(_bin_centers, _bin_n,
            width=(_bin_edges[1] - _bin_edges[0]) * 0.9,
            color='C7', alpha=0.3, label='sample count')
ax_twin.set_ylabel('# keypoints in bin', color='C7')
ax_twin.tick_params(axis='y', labelcolor='C7')

if _valid.any():
    ax_cal.plot(_bin_centers[_valid], _bin_mean[_valid],
                'o-', color='C1', linewidth=2, markersize=5, label='mean error')
    ax_cal.fill_between(_bin_centers[_valid], _bin_p25[_valid], _bin_p_hi[_valid],
                         color='C1', alpha=0.25,
                         label=f'25th–{ERROR_PERCENTILE}th pct.')

ax_cal.set_xlabel('predicted score')
ax_cal.set_ylabel(dist_unit)
ax_cal.set_title(f'Calibration curve — {_model_label_c}')
ax_cal.grid(True, alpha=0.3)
ax_cal.set_xlim(*_score_xlim)

_lines, _labels   = ax_cal.get_legend_handles_labels()
_bars,  _blabels  = ax_twin.get_legend_handles_labels()
ax_cal.legend(_lines + _bars, _labels + _blabels, fontsize=9, loc='upper right')

fig.tight_layout()
plt.show()

# ── Per-keypoint summary table ─────────────────────────────────────────────────
_id2name_c = _dmeta_c.get('keypoint_id2name', {})
_kpt_rows  = []
for _k in np.unique(kpt_ids):
    _m = kpt_ids == _k
    _s, _d = scores[_m], dists[_m]
    _rho, _pval = stats.spearmanr(_s, _d) if len(_s) > 2 else (np.nan, np.nan)
    _kpt_rows.append({
        'keypoint':   _id2name_c.get(str(_k), f'kpt_{_k}'),
        'n':          int(_m.sum()),
        'mean_score': float(_s.mean()),
        'mean_dist':  float(_d.mean()),
        'median_dist':float(np.median(_d)),
        'spearman_ρ': float(_rho),
        'p_value':    float(_pval),
    })

_kpt_df = (
    pd.DataFrame(_kpt_rows)
    .sort_values('mean_dist')
    .reset_index(drop=True)
)


def _rho_bg(series):
    """Green for ρ < 0 (higher score → lower error), red for ρ ≥ 0."""
    return [
        'background-color: #d4edda' if v < 0 else 'background-color: #f8d7da'
        for v in series
    ]


display(
    _kpt_df.style
    .apply(_rho_bg, subset=['spearman_ρ'])
    .format({
        'mean_score':  '{:.3f}',
        'mean_dist':   '{:.4f}',
        'median_dist': '{:.4f}',
        'spearman_ρ':  '{:+.3f}',
        'p_value':     '{:.2e}',
    }, na_rep='—')
    .set_caption(
        f'Per-keypoint calibration  |  {_model_label_c}  '
        f'|  ρ < 0 (green) = higher score → lower error  '
        f'|  dist unit: {dist_unit}'
    )
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '13px'), ('font-weight', 'bold')],
    }])
)